In [1]:
# 1. Tải Source code PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR

# 2. Cài đặt các thư viện phụ trợ
!pip install -r requirements.txt
!pip install rapidfuzz pyclipper lmdb

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 338449, done.
remote: Counting objects: 100% (1420/1420), done.
remote: Compressing objects: 100% (393/393), done.
remote: Total 338449 (delta 1193), reused 1027 (delta 1027), pack-reused 337029 (from 3)
Receiving objects: 100% (338449/338449), 1.79 GiB | 27.49 MiB/s, done.
Resolving deltas: 100% (268204/268204), done.
/content/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.8 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Cài đặt Kaggle API
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải và giải nén bộ ICDAR 2019 MLT
print("Đang tải dữ liệu 14GB từ Kaggle...")
!kaggle datasets download -d zubairalibhutto/mlt-19-ocr-dataset
print("Đang giải nén dữ liệu...")
!unzip -q mlt-19-ocr-dataset.zip -d /content/ICDAR2019

# Đường dẫn bộ dữ liệu gốc
raw_img_dir = '/content/ICDAR2019/TrainImages/TrainImages'
raw_gt_dir = '/content/ICDAR2019/TrainGT/TrainGT'

# Thư mục đích cho mô hình Recognition
rec_img_dir = '/content/PaddleOCR/train_data_rec/images'
rec_label_file = '/content/PaddleOCR/train_data_rec/rec_label.txt'
os.makedirs(rec_img_dir, exist_ok=True)

print("Bắt đầu xử lý và cắt (crop) ảnh tiếng Nhật...")
crop_count = 0

with open(rec_label_file, 'w', encoding='utf-8') as f_out:
    for gt_filename in os.listdir(raw_gt_dir):
        if not gt_filename.endswith('.txt'): continue

        gt_filepath = os.path.join(raw_gt_dir, gt_filename)
        img_filename = gt_filename.replace('.txt', '.jpg')
        if img_filename.startswith('gt_') and not os.path.exists(os.path.join(raw_img_dir, img_filename)):
            img_filename = img_filename.replace('gt_', '')

        img_filepath = os.path.join(raw_img_dir, img_filename)
        if not os.path.exists(img_filepath): continue

        img = cv2.imread(img_filepath)
        if img is None: continue

        # Đọc file nhãn gốc
        with open(gt_filepath, 'r', encoding='utf-8-sig') as f_in:
            for line in f_in:
                parts = line.strip().split(',')
                if len(parts) >= 10:
                    language = parts[8].strip()
                    text = ','.join(parts[9:]).strip()

                    # Chỉ lấy tiếng Nhật và bỏ qua nhãn không hợp lệ (###)
                    if language.lower() in ['japanese', 'jpn', 'english', 'latin'] and text != '###':
                        try:
                            # Lấy 4 điểm tọa độ
                            pts = np.array([
                                [int(parts[0]), int(parts[1])],
                                [int(parts[2]), int(parts[3])],
                                [int(parts[4]), int(parts[5])],
                                [int(parts[6]), int(parts[7])]
                            ], dtype=np.int32)

                            # Tính bounding box và cắt ảnh
                            x, y, w, h = cv2.boundingRect(pts)

                            # Bỏ qua nếu khung cắt nằm ngoài ảnh hoặc kích thước bằng 0
                            if x < 0 or y < 0 or w <= 0 or h <= 0: continue
                            crop_img = img[y:y+h, x:x+w]
                            if crop_img.size == 0: continue

                            # Lưu ảnh crop và ghi vào file label mới
                            crop_filename = f'crop_{crop_count:06d}.jpg'
                            cv2.imwrite(os.path.join(rec_img_dir, crop_filename), crop_img)
                            f_out.write(f"images/{crop_filename}\t{text}\n")
                            crop_count += 1
                        except ValueError:
                            pass

print(f"✅ Đã tạo thành công bộ dữ liệu Recognition với {crop_count} ảnh con!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang tải dữ liệu 14GB từ Kaggle...
Dataset URL: https://www.kaggle.com/datasets/zubairalibhutto/mlt-19-ocr-dataset
License(s): MIT
Resuming from 10571743232 bytes (3744497219 bytes left)...
100% 13.3G/13.3G [00:29<00:00, 125MB/s]

Đang giải nén dữ liệu...
Bắt đầu xử lý và cắt (crop) ảnh tiếng Nhật...
✅ Đã tạo thành công bộ dữ liệu Recognition với 64595 ảnh con!


In [3]:
import yaml
import os

config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 1. Cấu hình mô hình (TRAIN TỪ SCRATCH)
config['Global']['pretrained_model'] = None # Không dùng pre-trained
config['Global']['character_dict_path'] = 'ppocr/utils/dict/japan_dict.txt'
config['Global']['use_space_char'] = True

# 2. Cấu hình thư mục lưu về Google Drive
os.makedirs('/content/drive/MyDrive/Dict_OCR_Models', exist_ok=True)
config['Global']['save_model_dir'] = '/content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_v1'

# 3. Trỏ đường dẫn Dataset và tối ưu Batch Size
config['Train']['dataset']['data_dir'] = '/content/PaddleOCR/train_data_rec/'
config['Train']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data_rec/rec_label.txt']
config['Train']['loader']['batch_size_per_card'] = 64

config['Eval']['dataset']['data_dir'] = '/content/PaddleOCR/train_data_rec/'
config['Eval']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data_rec/rec_label.txt']
config['Eval']['loader']['batch_size_per_card'] = 64

# 4. Điều chỉnh Epoch cho Train Scratch
config['Global']['epoch_num'] = 50
config['Global']['save_epoch_step'] = 2
config['Global']['eval_batch_step'] = [0, 500]

# 5. Tự động nối tiếp huấn luyện (Auto-Resume)
save_dir = config['Global'].get('save_model_dir')
checkpoint_path = os.path.join(save_dir, 'latest')

if os.path.exists(f"{checkpoint_path}.pdparams"):
    config['Global']['checkpoints'] = checkpoint_path
    print(f"🔄 Sẽ Resume huấn luyện từ: {checkpoint_path}")
else:
    config['Global']['checkpoints'] = None

# 6. Các chốt chặn an toàn cho Colab
config['Train']['loader']['num_workers'] = 0
config['Eval']['loader']['num_workers'] = 0
config['Train']['loader']['use_shared_memory'] = False
config['Eval']['loader']['use_shared_memory'] = False
config['Global']['use_tensorrt'] = False
config['Global']['use_mkldnn'] = False

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f)

print("✅ Đã cập nhật file cấu hình YAML để Train từ Scratch!")

🔄 Sẽ Resume huấn luyện từ: /content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_v1/latest
✅ Đã cập nhật file cấu hình YAML để Train từ Scratch!


In [5]:
# Gỡ sạch các bản lỗi/xung đột hiện có
!python -m pip uninstall -y paddlepaddle paddlepaddle-gpu
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless

# Cài PaddlePaddle-GPU 2.6.1 (Bản tương thích CUDA 12)
!python -m pip install paddlepaddle-gpu==2.6.1.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

# Cài OpenCV Headless, cuDNN 8.9 và hạ cấp Numpy
!pip install opencv-python-headless==4.8.0.76
!pip install nvidia-cudnn-cu12==8.9.2.26
!pip install "numpy<2"

# Ép hệ thống nhận diện cuDNN cục bộ vừa cài
import os
import nvidia.cudnn
cudnn_path = os.path.dirname(nvidia.cudnn.__file__)
os.environ['LD_LIBRARY_PATH'] = f"{cudnn_path}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

print("✅ Đã xử lý xong môi trường: Paddle 2.6.1, OpenCV, cuDNN 8.9 và Numpy < 2")

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beha

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cudnn-cu12
    Found existing installation: nvidia-cudnn-cu12 9.10.2.21
    Uninstalling nvidia-cudnn-cu12-9.10.2.21:
      Successfully uninstalled nvidia-cudnn-cu12-9.10.2.21
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires nvidia-cudnn-cu12==9.10.2.21; platform_system == "Linux", but you have nvidia-cudnn-cu12 8.9.2.26 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the pac

✅ Đã xử lý xong môi trường: Paddle 2.6.1, OpenCV, cuDNN 8.9 và Numpy < 2


In [4]:
# 1. Tìm và tiêu diệt hoàn toàn cuDNN 9.8 của hệ điều hành
!rm -f /usr/lib/x86_64-linux-gnu/libcudnn*.so.9*
!rm -f /usr/local/cuda/lib64/libcudnn*.so.9*
!rm -f /usr/lib/x86_64-linux-gnu/libcudnn*.so

# 2. Tạo đường dẫn ép về bản 8.9
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn.so.8 /usr/lib/x86_64-linux-gnu/libcudnn.so
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_ops_infer.so.8 /usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_cnn_infer.so.8 /usr/lib/x86_64-linux-gnu/libcudnn_cnn_infer.so

!ldconfig

# 3. Chạy Train
%cd /content/PaddleOCR
!python tools/train.py -c configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbb

In [5]:
%cd /content/PaddleOCR
!python tools/export_model.py \
  -c configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml \
  -o Global.pretrained_model=/content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_v1/best_accuracy \
  Global.save_inference_dir=/content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_inference \
  Global.export_with_pir=False

/content/PaddleOCR
Skipping import of the encryption module.
W0511 08:56:40.526741 55536 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.0
W0511 08:56:40.527992 55536 gpu_resources.cc:164] device: 0, cuDNN Version: 8.9.
[2026/05/11 08:56:40] ppocr INFO: resume from /content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_v1/latest
[2026/05/11 08:56:41] ppocr INFO: Export inference config file to /content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_inference/inference.yml
Skipping import of the encryption module
I0511 08:56:42.378234 55536 program_interpreter.cc:212] New Executor is Running.
[2026/05/11 08:56:42] ppocr INFO: inference model is saved to /content/drive/MyDrive/Dict_OCR_Models/rec_japan_scratch_inference/inference
